# 03 – Imputation Methods Comparison

Apply all deterministic and interpolation-based imputation methods to a 30%-gapped signal and compare them visually.


In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from src.simulation.generator import generate_synthetic_lightcurve
from src.missingness.injector import inject_gaps
from src.imputation.registry import get_imputer
from src.evaluation.metrics import compute_rmse

t, flux, params = generate_synthetic_lightcurve(N=500, seed=0)
gapped, missing_idx, true_vals = inject_gaps(flux, p=0.30, seed=42)

# Run deterministic methods
methods_to_show = ['mean_fill', 'forward_fill', 'linear_interp', 'spline_interp']
results = {}
for key in methods_to_show:
    imp = get_imputer(key)
    imputed = imp.impute(t, gapped, missing_idx, period_est=1.0)
    rmse = compute_rmse(true_vals, imputed[missing_idx])
    results[imp.name] = (imputed, rmse)
    print(f'{imp.name:25s}  RMSE={rmse:.5f}')

In [ ]:
# Zoom into first 60 days
zoom = t < 6.0
fig, axes = plt.subplots(len(results), 1, figsize=(12, 10), sharex=True)

obs_mask = ~np.isnan(gapped)
colors_map = {'Mean-Fill': '#e41a1c', 'Forward-Fill': '#ff7f00', 'Linear-Interp': '#377eb8', 'Spline-Interp': '#4daf4a'}

for ax, (name, (imputed, rmse)) in zip(axes, results.items()):
    ax.plot(t[zoom & obs_mask], flux[zoom & obs_mask], 'k.', ms=2, label='Observed', zorder=5)
    ax.plot(t[zoom], flux[zoom], '--', lw=0.8, color='grey', alpha=0.5, label='Ground truth')
    ax.plot(t[zoom], imputed[zoom], lw=1.2, color=colors_map.get(name, 'steelblue'), label=f'{name} (RMSE={rmse:.4f})')
    ax.legend(loc='upper right', fontsize=8)
    ax.set_ylabel('Flux')
    ax.grid(alpha=0.2)

axes[-1].set_xlabel('Time (days)')
plt.suptitle('Deterministic imputation methods — first 6 days', fontsize=12)
plt.tight_layout()
plt.show()